# DATA3404 Assignment 2 - Scaffold Code

This is the SQL template notebook for the second assignment on Databricks, 2025s1.

This notebook assumes that you have executed the **Bootstrap** notebook (e.g. from Assignment 1) first in your Databricks account.
You should run that Bootstrap code only once. It downloads and installs the CSV files of our dataset which we use in Assignment 2 too.

### Check whether schema already exists
Brief check whether we have the SQL schemas already existing in current workspace, e.g. left from Assignment 1.  
Expected table sizes:  
<style scoped>
table {
  font-size: 10px;
}
</style>
| table         |  count  |
|---------------|--------:|
|Cities         |      12 |
|Neighbourhoods |     551 |
|Hosts          |   61153 |
|Listings_small |   10500 |
|Listings_medium|   54000 |
|Listings_large |  108182 |
|Reviews_small  |  400000 |
|Reviews_medium | 2000000 |
|Reviews_large  | 4009676 |


In [0]:
%sql
SELECT 'Cities', COUNT(*) FROM Cities
UNION
SELECT 'Neighbourhoods', COUNT(*) FROM Neighbourhoods
UNION
SELECT 'Hosts', COUNT(*) FROM Hosts
UNION
SELECT 'Listings_small', COUNT(*) FROM Listings_small
UNION
SELECT 'Listings_medium', COUNT(*) FROM Listings_medium
UNION
SELECT 'Listings_large', COUNT(*) FROM Listings_large
UNION
SELECT 'Reviews_small', COUNT(*) FROM Reviews_small
UNION
SELECT 'Reviews_medium', COUNT(*) FROM Reviews_medium
UNION
SELECT 'Reviews_large', COUNT(*) FROM Reviews_large


Cities,count(1)
Cities,12
Neighbourhoods,551
Hosts,61153
Listings_small,10500
Listings_medium,54000
Listings_large,108182
Reviews_small,400000
Reviews_medium,2000000
Reviews_large,4009676


### Create SQL Schema for files<br>ONLY RUN THIS if you get an '[TABLE_OR_VIEW_NOT_FOUND]' error on the check above
Only run this if you try out the original SQl query from Task 1 and if ou get an '[TABLE_OR_VIEW_NOT_FOUND]' error on the table size check above.

The following will map then the imported CSV files as SQL tables so that they are available for subsequent SQL queries using Hive or pySpark with a number of tables:
 **Cities**, **Neighbourhoods**, **Hosts** and three variants of the
 **Listings_**_scale_ and the **Reviews_**_scale_  tables. You can switch between differenty dataset sizes by
referring to _either_ the **Listings_small**, _or_ **Listings_medium** _or_
 **Listings_large** tables, and reespectively for the **Reviews_**_scale_  tables.

In [0]:
%sql
DROP TABLE IF EXISTS Cities;
CREATE TABLE Cities (
  id             INTEGER,
  city_name      VARCHAR(20),
  state          VARCHAR(40),
  country        VARCHAR(40),
  country_code   CHAR(2)
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_cities.csv", header "true");

DROP TABLE IF EXISTS Neighbourhoods;
CREATE TABLE Neighbourhoods (
  id             INTEGER,
  city_id        INTEGER,
  nhood_name     VARCHAR(50),
  nhood_group    VARCHAR(50),
  geometry       BINARY
) 
USING csv
OPTIONS (path "/FileStore/tables/airbnb_neighbourhoods.csv", header "true");

DROP TABLE IF EXISTS Hosts;
CREATE TABLE Hosts (
  id              INTEGER,
  host_name       VARCHAR(50),
  host_since      DATE,
  host_about      VARCHAR(1000),
  is_superhost    CHAR(1),
  response_time   VARCHAR(20),
  response_rate   VARCHAR(4),
  acceptance_rate VARCHAR(4),
  last_scraped    DATE
) 
USING csv
OPTIONS (path "/FileStore/tables/airbnb_hosts.csv", header "true");

DROP TABLE IF EXISTS Listings_small;
CREATE TABLE Listings_Small (
  id             BIGINT,
  listing_name   VARCHAR(250),
  property_type  VARCHAR(40),
  room_type      VARCHAR(15),
  price          FLOAT,
  minimum_nights INTEGER,
  host_id        INTEGER,
  city_id        INTEGER,
  neighbourhood  INTEGER,
  latitude       NUMERIC(9,6),
  longitude      NUMERIC(9,6),
  description    VARCHAR(1000),
  accommodates   INT,
  bathrooms      INT,
  bedrooms       INT,
  beds           INT,
  amenities      STRING, -- ARRAY<STRING>
  last_scraped   DATE
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_listings-small.csv", header "true");

DROP TABLE IF EXISTS Listings_medium;
CREATE TABLE Listings_Medium (
  id             BIGINT,
  listing_name   VARCHAR(250),
  property_type  VARCHAR(40),
  room_type      VARCHAR(15),
  price          FLOAT,
  minimum_nights INTEGER,
  host_id        INTEGER,
  city_id        INTEGER,
  neighbourhood  INTEGER,
  latitude       NUMERIC(9,6),
  longitude      NUMERIC(9,6),
  description    VARCHAR(1000),
  accommodates   INT,
  bathrooms      INT,
  bedrooms       INT,
  beds           INT,
  amenities      STRING, -- ARRAY<STRING>,
  last_scraped   DATE
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_listings-medium.csv", header "true");

DROP TABLE IF EXISTS Listings_large;
CREATE TABLE Listings_Large (
  id             BIGINT,
  listing_name   VARCHAR(250),
  property_type  VARCHAR(40),
  room_type      VARCHAR(15),
  price          FLOAT,
  minimum_nights INTEGER,
  host_id        INTEGER,
  city_id        INTEGER,
  neighbourhood  INTEGER,
  latitude       NUMERIC(9,6),
  longitude      NUMERIC(9,6),
  description    VARCHAR(1000),
  accommodates   INT,
  bathrooms      INT,
  bedrooms       INT,
  beds           INT,
  amenities      STRING, -- ARRAY<STRING>,
  last_scraped   DATE
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_listings-large.csv", header "true");

DROP TABLE IF EXISTS Reviews_small;
CREATE TABLE Reviews_small (
  id             BIGINT,
  listing_id     BIGINT,
  review_date    DATE    NOT NULL,
  reviewer_id    INTEGER NOT NULL,
  reviewer_name  VARCHAR(50),
  comments       STRING
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_reviews-small.csv", header "true");

DROP TABLE IF EXISTS Reviews_medium;
CREATE TABLE Reviews_medium (
  id             BIGINT,
  listing_id     BIGINT,
  review_date    DATE    NOT NULL,
  reviewer_id    INTEGER NOT NULL,
  reviewer_name  VARCHAR(50),
  comments       STRING
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_reviews-medium.csv", header "true");

DROP TABLE IF EXISTS Reviews_large;
CREATE TABLE Reviews_large (
  id             BIGINT,
  listing_id     BIGINT,
  review_date    DATE    NOT NULL,
  reviewer_id    INTEGER NOT NULL,
  reviewer_name  VARCHAR(50),
  comments       STRING
)
USING csv
OPTIONS (path "/FileStore/tables/airbnb_reviews-large.csv", header "true");


Install Packages before run code

In [0]:
%pip install sparkmeasure

Python interpreter will be restarted.
Python interpreter will be restarted.


<hr>

## Task 1:  SQL to pySpark
Rewrite the following SQL query as Python Spark code, using its Python Dataframe API:

In [0]:
%sql
SELECT L.listing_name, C.city_name, COUNT(DISTINCT R.id) AS num_reviews
  FROM Hosts H JOIN Listings_Large L  ON (L.host_id=H.id)
               JOIN Reviews_Large R   ON (R.listing_id=L.id)
               JOIN Cities C ON (L.city_id=C.id)
 WHERE H.is_superhost = 't'
   AND L.room_type = 'Entire home/apt'
   AND EXTRACT(year FROM R.review_date) = 2025
   AND C.country_code = 'AU'
 GROUP BY L.listing_name, C.city_name
 ORDER BY num_reviews DESC, L.listing_name
 LIMIT 5;

listing_name,city_name,num_reviews
One-Bedroom Apartment,Melbourne,53
Blessington St Studio Apartments,Melbourne,44
Brand New Studio Apartment in Sunshine,Melbourne,37
"Lux Beach Retreat, 2 beds, fire-pit, ensuite, gym!",Sydney,30
St Kilda stunner - rooftop infinity pool + parking,Melbourne,28


### Next, we will run the small, medium and large tables of Reviews and listing respectively to check that the calculation time of pyspark code running without optimization is 12s/29s/50s respectively. The larger the table, the more data there is, and the longer the running calculation time will be

In [0]:
# Disable auto-optimisation for consistency
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)

In [0]:
#Small
hosts_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_hosts.csv")
)

listings_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_listings-small.csv")
)

reviews_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_reviews-small.csv")
)

cities_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_cities.csv")
)

from pyspark.sql import functions as F

result_df = (
    hosts_df.alias("H")
      .join(listings_df.alias("L"), F.col("L.host_id") == F.col("H.id"))
      .join(reviews_df.alias("R"), F.col("R.listing_id") == F.col("L.id"))
      .join(cities_df.alias("C"), F.col("L.city_id") == F.col("C.id"))
      .filter((F.col("H.is_superhost") == "t") &
              (F.col("L.room_type") == "Entire home/apt") &
              (F.year(F.col("R.review_date")) == 2025) &
              (F.col("C.country_code") == "AU"))
      .groupBy(F.col("L.listing_name"), F.col("C.city_name"))
      .agg(F.countDistinct(F.col("R.id")).alias("num_reviews"))
      .orderBy(F.col("num_reviews").desc(), F.col("L.listing_name"))
      .limit(5)
)

result_df.show()

+--------------------+---------+-----------+
|        listing_name|city_name|num_reviews|
+--------------------+---------+-----------+
|One-Bedroom Apart...|Melbourne|         53|
|South Brisbane Ci...| Brisbane|         26|
|Wanderlust - I wa...|Melbourne|         24|
|1br near airport ...|   Sydney|         22|
|Carlton Luxurious...|Melbourne|         19|
+--------------------+---------+-----------+



In [0]:
#medium
hosts_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_hosts.csv")
)

listings_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_listings-medium.csv")
)

reviews_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_reviews-medium.csv")
)

cities_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_cities.csv")
)

from pyspark.sql import functions as F

result_df = (
    hosts_df.alias("H")
      .join(listings_df.alias("L"), F.col("L.host_id") == F.col("H.id"))
      .join(reviews_df.alias("R"), F.col("R.listing_id") == F.col("L.id"))
      .join(cities_df.alias("C"), F.col("L.city_id") == F.col("C.id"))
      .filter((F.col("H.is_superhost") == "t") &
              (F.col("L.room_type") == "Entire home/apt") &
              (F.year(F.col("R.review_date")) == 2025) &
              (F.col("C.country_code") == "AU"))
      .groupBy(F.col("L.listing_name"), F.col("C.city_name"))
      .agg(F.countDistinct(F.col("R.id")).alias("num_reviews"))
      .orderBy(F.col("num_reviews").desc(), F.col("L.listing_name"))
      .limit(5)
)

result_df.show()


+--------------------+---------+-----------+
|        listing_name|city_name|num_reviews|
+--------------------+---------+-----------+
|One-Bedroom Apart...|Melbourne|         53|
|Brand New Studio ...|Melbourne|         37|
|Lux Beach Retreat...|   Sydney|         30|
|    Bayside Bungalow|Melbourne|         25|
|Miss Baker's Bond...|   Sydney|         25|
+--------------------+---------+-----------+



In [0]:
from pyspark.sql import functions as F
# Disable auto-optimisation for consistency
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)
#Large
from sparkmeasure import StageMetrics
stagemetrics = StageMetrics(spark)
stagemetrics.begin()

spark.sql("CLEAR CACHE").collect()


hosts_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_hosts.csv")
)

listings_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_listings-large.csv")
)

reviews_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_reviews-large.csv")
)

cities_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_cities.csv")
)

result_df = (
    hosts_df.alias("H")
      .join(listings_df.alias("L"), F.col("L.host_id") == F.col("H.id"))
      .join(reviews_df.alias("R"), F.col("R.listing_id") == F.col("L.id"))
      .join(cities_df.alias("C"), F.col("L.city_id") == F.col("C.id"))
      .filter((F.col("H.is_superhost") == "t") &
              (F.col("L.room_type") == "Entire home/apt") &
              (F.year(F.col("R.review_date")) == 2025) &
              (F.col("C.country_code") == "AU"))
      .groupBy(F.col("L.listing_name"), F.col("C.city_name"))
      .agg(F.countDistinct(F.col("R.id")).alias("num_reviews"))
      .orderBy(F.col("num_reviews").desc(), F.col("L.listing_name"))
      .limit(5)
)

result_df.show()


# This prints a report of aggregated metrics values
stagemetrics.end()
stagemetrics.print_report()


+--------------------+---------+-----------+
|        listing_name|city_name|num_reviews|
+--------------------+---------+-----------+
|One-Bedroom Apart...|Melbourne|         53|
|Blessington St St...|Melbourne|         44|
|Brand New Studio ...|Melbourne|         37|
|Lux Beach Retreat...|   Sydney|         30|
|St Kilda stunner ...|Melbourne|         28|
+--------------------+---------+-----------+


Scheduling mode = FAIR
Spark Context default degree of parallelism = 8

Aggregated Spark stage metrics:
numStages => 15
numTasks => 642
elapsedTime => 99315 (1.7 min)
stageDuration => 111280 (1.9 min)
executorRunTime => 536456 (8.9 min)
executorCpuTime => 80977 (1.3 min)
executorDeserializeTime => 118869 (2.0 min)
executorDeserializeCpuTime => 21073 (21 s)
resultSerializationTime => 754 (0.8 s)
jvmGCTime => 12468 (12 s)
shuffleFetchWaitTime => 143 (0.1 s)
shuffleWriteTime => 52298 (52 s)
resultSize => 833252 (813.7 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
pe

<hr>

## Task 2:  Optimisation and Physical Design alternatives
Rewrite the following SQL query as Python Spark code, using its Python Dataframe API:

In [0]:
# Part1 excution plan with no optimization
result_df.explain(True)  

== Parsed Logical Plan ==
GlobalLimit 5
+- LocalLimit 5
   +- Sort [num_reviews#1406L DESC NULLS LAST, listing_name#1077 ASC NULLS FIRST], true
      +- Aggregate [listing_name#1077, city_name#1159], [listing_name#1077, city_name#1159, count(distinct id#1129L) AS num_reviews#1406L]
         +- Filter ((((is_superhost#1045 = t) AND (room_type#1079 = Entire home/apt)) AND (year(review_date#1131) = 2025)) AND (country_code#1162 = AU))
            +- Join Inner, (city_id#1083 = cast(id#1158 as double))
               :- Join Inner, (listing_id#1130L = id#1076L)
               :  :- Join Inner, (cast(host_id#1082 as int) = id#1041)
               :  :  :- SubqueryAlias H
               :  :  :  +- Relation [id#1041,host_name#1042,host_since#1043,host_about#1044,is_superhost#1045,response_time#1046,response_rate#1047,acceptance_rate#1048,last_scraped#1049] csv
               :  :  +- SubqueryAlias L
               :  :     +- Relation [id#1076L,listing_name#1077,property_type#1078,room_type#

### Code Optimize
code optimization focus on：
1. broadcast on small table（hosts，cities）
2. Only select some of the features
3. Cache, subsequent join reuse
4. Screen the conditions in advance

In [0]:
# Disable auto-optimisation for consistency
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)

# Part 2 Code optimization
from pyspark.sql import functions as F

hosts_opt = (
    hosts_df
      .filter(F.col("is_superhost") == "t")        
      .select("id")                                 
      .hint("broadcast")                            
)

cities_opt = (
    cities_df
      .filter(F.col("country_code") == "AU")        
      .select("id", "city_name")                    
      .hint("broadcast")                           
)

listings_opt = (
    listings_df
      .filter(F.col("room_type") == "Entire home/apt")  
      .select("id", "listing_name", "host_id", "city_id")
)

reviews_opt = (
    reviews_df
      .filter(F.year("review_date") == 2025)      
      .select("id", "listing_id")                  
      .cache()                                     
)

opt_df = (
    hosts_opt.alias("H")
      .join(listings_opt.alias("L"), F.col("L.host_id") == F.col("H.id"))
      .join(reviews_opt.alias("R"), F.col("R.listing_id") == F.col("L.id"))
      .join(cities_opt.alias("C"), F.col("L.city_id") == F.col("C.id"))
      .groupBy("L.listing_name", "C.city_name")
      .agg(F.countDistinct("R.id").alias("num_reviews"))
      .orderBy(F.desc("num_reviews"), "L.listing_name")
      .limit(5)
)

opt_df.show()

opt_df.explain(True)



+--------------------+---------+-----------+
|        listing_name|city_name|num_reviews|
+--------------------+---------+-----------+
|One-Bedroom Apart...|Melbourne|         53|
|Blessington St St...|Melbourne|         44|
|Brand New Studio ...|Melbourne|         37|
|Lux Beach Retreat...|   Sydney|         30|
|St Kilda stunner ...|Melbourne|         28|
+--------------------+---------+-----------+

== Parsed Logical Plan ==
GlobalLimit 5
+- LocalLimit 5
   +- Sort [num_reviews#9519L DESC NULLS LAST, listing_name#9071 ASC NULLS FIRST], true
      +- Aggregate [listing_name#9071, city_name#9153], [listing_name#9071, city_name#9153, count(distinct id#9123L) AS num_reviews#9519L]
         +- Join Inner, (city_id#9077 = cast(id#9152 as double))
            :- Join Inner, (listing_id#9124L = id#9070L)
            :  :- Join Inner, (cast(host_id#9076 as int) = id#9035)
            :  :  :- SubqueryAlias H
            :  :  :  +- ResolvedHint (strategy=broadcast)
            :  :  :     +-

In [0]:
# Part 3 Creat Parquet file
from pyspark.sql import SparkSession, functions as F


# 1. Convert CSV to Parquet (at one time) and write it to the tables_parquet directory
spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_hosts.csv") \
     .write.mode("overwrite") \
     .parquet(f"dbfs:/FileStore/tables_parquet/airbnb_hosts.parquet")

spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_listings-large.csv") \
     .write.mode("overwrite") \
     .parquet(f"dbfs:/FileStore/tables_parquet/airbnb_listings-large.parquet")

spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_cities.csv") \
     .write.mode("overwrite") \
     .parquet(f"dbfs:/FileStore/tables_parquet/airbnb_cities.parquet")

# reviews Write by partition by year
spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_reviews-large.csv") \
     .withColumn("review_date", F.to_date("review_date", "yyyy-MM-dd")) \
     .withColumn("yr", F.year("review_date")) \
     .write \
         .mode("overwrite") \
         .partitionBy("yr") \
         .parquet("dbfs:/FileStore/tables_parquet/airbnb_reviews_by_year-large.parquet")


In [0]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)
stagemetrics.begin()
spark.sql("CLEAR CACHE").collect()


# Part 3 Parquet file：This part begins the formal data processing and time calculation
hosts_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_hosts.parquet")
cities_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_cities.parquet")
listings_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_listings-large.parquet")
reviews_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_reviews_by_year-large.parquet/yr=2025")# Only the 2025 partition is available

# Push down in advance for filtering & column cropping & broadcasting
hosts_opt    = (hosts_p.filter("is_superhost = 't'")
                .select("id")
                .hint("broadcast")
                )
cities_opt   = (cities_p
                .filter("country_code = 'AU'")
                .select("id","city_name")
                .hint("broadcast")
                )
listings_opt = (listings_p
                .filter("room_type = 'Entire home/apt'")
                .select("id","listing_name","host_id","city_id"))
reviews_opt  = reviews_p.select("id","listing_id") 

opt_parquet_df = (
    hosts_opt.alias("H")
      .join(listings_opt.alias("L"), F.col("L.host_id") == F.col("H.id"))
      .join(reviews_opt.alias("R"), F.col("R.listing_id") == F.col("L.id"))
      .join(cities_opt.alias("C"), F.col("L.city_id") == F.col("C.id"))
      .groupBy("L.listing_name","C.city_name")
      .agg(F.countDistinct("R.id").alias("num_reviews"))
      .orderBy(F.desc("num_reviews"), "L.listing_name")
      .limit(5)
)

opt_parquet_df.show()
opt_parquet_df.explain(True)

# This prints a report of aggregated metrics values
stagemetrics.end()
stagemetrics.print_report()

+--------------------+---------+-----------+
|        listing_name|city_name|num_reviews|
+--------------------+---------+-----------+
|One-Bedroom Apart...|Melbourne|         53|
|Blessington St St...|Melbourne|         44|
|Brand New Studio ...|Melbourne|         37|
|Lux Beach Retreat...|   Sydney|         30|
|St Kilda stunner ...|Melbourne|         28|
+--------------------+---------+-----------+

== Parsed Logical Plan ==
GlobalLimit 5
+- LocalLimit 5
   +- Sort [num_reviews#9832L DESC NULLS LAST, listing_name#9721 ASC NULLS FIRST], true
      +- Aggregate [listing_name#9721, city_name#9711], [listing_name#9721, city_name#9711, count(distinct id#9756) AS num_reviews#9832L]
         +- Join Inner, (city_id#9727 = id#9710)
            :- Join Inner, (listing_id#9757 = id#9720)
            :  :- Join Inner, (host_id#9726 = id#9692)
            :  :  :- SubqueryAlias H
            :  :  :  +- ResolvedHint (strategy=broadcast)
            :  :  :     +- Project [id#9692]
            :

<hr>

## Task 3: Data Analysis in pySpark
Write pySpark code for the problem specification from Assignment 2, Task 3, and analyse its scalability with and without optimisations.

In [0]:
%pip install sparkmeasure

Python interpreter will be restarted.
  Created wheel for pyspark: filename=pyspark-4.0.0-py2.py3-none-any.whl size=434741259 sha256=dc05afc8bb8d5b34817a3198986aa3d90aac5bfc836d2c9a1bc008b08ed84052
  Stored in directory: /root/.cache/pip/wheels/11/95/4b/e3cfd6f160b2988f57207fcde6e5a4c6e0104d7da5a107eb8e
Successfully built pyspark
Python interpreter will be restarted.


### Without optimization

In [0]:
#Without optimisation
from sparkmeasure import StageMetrics
stagemetrics = StageMetrics(spark)

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)
spark.catalog.clearCache()
stagemetrics.begin()


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# ==================You can defind the city here===================
city_name = "Sydney"
# ==================You can defind the city here===================


# Step 1 : read file
cities_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_cities.csv")
         .hint("merge")
)
listings_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_listings-large.csv") #======= You can change dataset here('-medium'/'-small') ======
         .hint("merge")
)
reviews_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_reviews-large.csv") #======= You can change dataset here('-medium'/'-small') ======
         .hint("merge")
)
neighbourhoods_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_neighbourhoods.csv")
         .hint("merge")
)

# Step 2 : Screen the properties in the selected area (Sydney) and associate them with the community name
select_city_id = (cities_df
                  .filter(F.col("city_name") == city_name)
                  )
# Based on the city information jion form we just found, form a complete listing form including the Sydney area, streets and check-in information
listings_select = (listings_df
                .join(select_city_id, listings_df["city_id"] == select_city_id["id"])
                .join(neighbourhoods_df, listings_df["neighbourhood"] == neighbourhoods_df["id"]) 
                .drop(neighbourhoods_df["city_id"],neighbourhoods_df["id"],select_city_id['id']) 
                .withColumnRenamed("nhood_name", "neighbourhood_name") 
                .withColumnRenamed("id", "listing_id")
)
# Step3: Filter out the reviews for 2024, associate the reviews with the property data (retain only the reviews for properties in Sydney).

reviews_select_2024 = (reviews_df.join(listings_select, on="listing_id", how="inner")
                    .withColumn("review_date", F.to_date("review_date", "yyyy/M/d"))
                    .filter(F.year("review_date") == 2024)
                    )
# The inner join retains records in the intersection of the two tables - that is, only those rows where the listing_id has appeared on both sides. In this way, we can discard reviews that do not belong to Sydney, discard properties in Sydney that have no reviews, and only retain the part of the data that we truly need, which is "Sydney has reviews in 2024"

# Calculate the number of days of stay (occ_days) corresponding to each comment according to the rules

reviews_select_2024 = (reviews_select_2024
                    .withColumn("stay_length",
                                F.when((F.col("minimum_nights") > 3) & (F.col("minimum_nights") <= 21), F.col("minimum_nights"))
                                .when(F.col("minimum_nights") > 21, F.lit(3))
                                .otherwise(F.lit(3)))
                    .withColumn("occupation_days", F.col("stay_length") * F.lit(2))
                    )

# Group the review data with the property ID by listing_id, sum up occupation_days to obtain the total number of occupancy days for each property in 2024, occupation_days_sum

listing_occupation = (reviews_select_2024
               .groupBy("listing_id") 
               .agg(F.sum("occupation_days").alias("occupation_days_sum"))
               )

# Merge the total number of stay days with the list of all listings. Fill in 0 for missing values (no comments will be regarded as 0 stays).

listings_occupation = (listings_select.join(listing_occupation, on="listing_id", how="left") 
                       .fillna({"occupation_days_sum": 0})
                       )

# Calculate the occupancy rate of each property as the number of occupied days divided by 365, with a cap of no more than 1
listings_occupation = (listings_occupation.withColumn("occupancy_rate",
                                                      F.least(F.col("occupation_days_sum") / F.lit(365), F.lit(1.0))
                                                      )
                       )

# Calculate the average occupancy rate for each community (grouped by neighbourhood_name, take the average of occupancy y_rate for all properties in the community, in descending order of average occupancy rate, only the top 10 communities.) Select the top 10 communities in descending order of average occupancy rate and arrange them in descending order

top10_neigh = (listings_occupation
             .groupBy("neighbourhood_name") 
             .agg(F.avg("occupancy_rate").alias("average_occupancy_rate"))
             .orderBy(F.col("average_occupancy_rate").desc())
             .limit(10)
             )

from pyspark.sql.window import Window

# Rank the housing resources within each community by occupancy_rate, select the top 5 streets, and retain rn
window = Window.partitionBy("neighbourhood_name").orderBy(F.col("occupancy_rate").desc())
# Based on the window defined above, use row_number() to type the consecutive ranking rn for the rows within each group
ranked = listings_occupation.withColumn("rn", F.row_number().over(window))

top5_listings = (ranked.filter(F.col("rn") <= 5)
                 .select("neighbourhood_name", "listing_name", "occupancy_rate", "rn")
                 .join(top10_neigh.select("neighbourhood_name"), on="neighbourhood_name")
                 )

# List of the top 5 properties aggregated by community
top5_grouped = (top5_listings
                .groupBy("neighbourhood_name")
                .agg(F.collect_list(F.struct(F.col("rn"), F.col("listing_name"), F.col("occupancy_rate"))).alias("listings_struct"))
                .join(top10_neigh, on="neighbourhood_name")
                .select("neighbourhood_name","average_occupancy_rate","listings_struct")
                .orderBy(F.desc("average_occupancy_rate"),F.desc("neighbourhood_name"))
                )  

# show result
df1 = top5_grouped.withColumn(
    "pairs",
    F.transform(
        F.col("listings_struct"),
        lambda x: F.concat(
            F.lit("("),
            x["listing_name"],
            F.lit(", "),
            x["occupancy_rate"].cast("string"),
            F.lit(")")
        )
    )
)

df2 = df1.withColumn(
    "listings_str",
    F.concat(
        F.lit("["),
        F.concat_ws(", ", F.col("pairs")),
        F.lit("]")
    )
)

opt_n_1 = df2.withColumn("avg_occ_rate_str", F.format_number(F.col("average_occupancy_rate"), 2))
opt_n_1 .select(F.concat_ws("\t", F.col("neighbourhood_name"), F.col("avg_occ_rate_str"), F.col("listings_str")).alias("output")).show(truncate=False)




# This prints a report of aggregated metrics values
opt_n_1 .explain(True)
stagemetrics.end()
stagemetrics.print_report()

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|output                                                                                                                                                                                                                                                                                                                                  |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Sydney

### Code optimization

In [0]:
from sparkmeasure import StageMetrics
stagemetrics = StageMetrics(spark)

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)
spark.catalog.clearCache()
stagemetrics.begin()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# ==================You can defind the city here===================
city_name = "Sydney"
# ==================You can defind the city here===================


cities_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_cities.csv")
         .select("id","city_name")
)
listings_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_listings-large.csv")
         .select("id","city_id","neighbourhood","listing_name","minimum_nights")
)
reviews_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_reviews-large.csv")
         .select("listing_id","review_date")
)
neighbourhoods_df = (
    spark.read.format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/FileStore/tables/airbnb_neighbourhoods.csv")
         .select("id","nhood_name","city_id")
)


select_city_id = (cities_df
                  .filter(F.col("city_name") == city_name)
                  )
listings_select = (listings_df
                .join(select_city_id, listings_df["city_id"] == select_city_id["id"])
                .join(F.broadcast(neighbourhoods_df), listings_df["neighbourhood"] == neighbourhoods_df["id"]) 
                .drop(neighbourhoods_df["city_id"],neighbourhoods_df["id"],select_city_id['id']) 
                .withColumnRenamed("nhood_name", "neighbourhood_name") 
                .withColumnRenamed("id", "listing_id")
)

reviews_2024 = (reviews_df
                .withColumn("review_date", F.to_date("review_date", "yyyy/M/d"))
                .filter(F.year("review_date") == 2024)
                .select("listing_id")  
                .cache()
)
reviews_select_2024 = reviews_2024.join(listings_select, on="listing_id", how="inner")

reviews_select_2024 = (reviews_select_2024
                    .withColumn("stay_length",
                                F.when((F.col("minimum_nights") > 3) & (F.col("minimum_nights") <= 21), F.col("minimum_nights"))
                                .when(F.col("minimum_nights") > 21, F.lit(3))
                                .otherwise(F.lit(3)))
                    .withColumn("occupation_days", F.col("stay_length") * F.lit(2))
                    )


listings_occupation = (listings_select
                       .join(reviews_select_2024
                             .groupBy("listing_id")
                             .agg(F.sum("occupation_days").alias("occupation_days_sum")),
                             on="listing_id",
                             how="left"
                             )
                       .fillna({"occupation_days_sum": 0})
                       .withColumn("occupancy_rate",
                                   F.least(F.col("occupation_days_sum") / F.lit(365), F.lit(1.0))
                                   )
                       )


top10_neigh = (listings_occupation
             .groupBy("neighbourhood_name") 
             .agg(F.avg("occupancy_rate").alias("average_occupancy_rate"))
             .orderBy(F.col("average_occupancy_rate").desc())
             .limit(10)
             )

from pyspark.sql.window import Window


window = Window.partitionBy("neighbourhood_name").orderBy(F.col("occupancy_rate").desc())

top5_grouped = (listings_occupation
                .join(F.broadcast(top10_neigh), on="neighbourhood_name", how="inner")
                .withColumn("rn", F.row_number().over(window))
                .filter(F.col("rn") <= 5)
                .groupBy("neighbourhood_name", "average_occupancy_rate")
                .agg(F.collect_list(F.struct("rn", "listing_name", "occupancy_rate")).alias("listings_struct"))
                .orderBy(F.desc("average_occupancy_rate"),F.desc("neighbourhood_name"))
                )

df1 = top5_grouped.withColumn(
    "pairs",
    F.transform(
        F.col("listings_struct"),
        lambda x: F.concat(
            F.lit("("),
            x["listing_name"],
            F.lit(", "),
            x["occupancy_rate"].cast("string"),
            F.lit(")")
        )
    )
)

df2 = df1.withColumn(
    "listings_str",
    F.concat(
        F.lit("["),
        F.concat_ws(", ", F.col("pairs")),
        F.lit("]")
    )
)

opt_c_2 = df2.withColumn("avg_occ_rate_str", F.format_number(F.col("average_occupancy_rate"), 2))


opt_c_2.select(F.concat_ws("\t", F.col("neighbourhood_name"), F.col("avg_occ_rate_str"), F.col("listings_str")).alias("output")).show(truncate=False)


opt_c_2.explain(True)

stagemetrics.end()
stagemetrics.print_report()

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|output                                                                                                                                                                                                                                                                                                                                  |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Sydney

### With Physical Optimisation

In [0]:
spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_neighbourhoods.csv") \
     .write.mode("overwrite") \
     .parquet(f"dbfs:/FileStore/tables_parquet/airbnb_neighbourhoods.parquet")


spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_listings-medium.csv") \
     .write.mode("overwrite") \
     .parquet(f"dbfs:/FileStore/tables_parquet/airbnb_listings-medium.parquet")

spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_listings-small.csv") \
     .write.mode("overwrite") \
     .parquet(f"dbfs:/FileStore/tables_parquet/airbnb_listings-small.parquet")

spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_reviews-medium.csv") \
     .withColumn("review_date", F.to_date("review_date", "yyyy-MM-dd")) \
     .withColumn("yr", F.year("review_date")) \
     .write \
         .mode("overwrite") \
         .partitionBy("yr") \
         .parquet("dbfs:/FileStore/tables_parquet/airbnb_reviews_by_year-medium.parquet")

spark.read.format("csv").option("header","true") \
     .load("dbfs:/FileStore/tables/airbnb_reviews-small.csv") \
     .withColumn("review_date", F.to_date("review_date", "yyyy-MM-dd")) \
     .withColumn("yr", F.year("review_date")) \
     .write \
         .mode("overwrite") \
         .partitionBy("yr") \
         .parquet("dbfs:/FileStore/tables_parquet/airbnb_reviews_by_year-small.parquet")

In [0]:
from sparkmeasure import StageMetrics
stagemetrics = StageMetrics(spark)
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.cbo.enabled", False)
spark.catalog.clearCache()
stagemetrics.begin()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# ==================You can defind the city here===================
city_name = "Sydney"
# ==================You can defind the city here===================

neighbourhoods_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_neighbourhoods.parquet")
cities_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_cities.parquet")
listings_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_listings-large.parquet")
reviews_p = spark.read.parquet(f"dbfs:/FileStore/tables_parquet/airbnb_reviews_by_year-large.parquet/yr=2024")# Only the 2024 partition is available

neighbourhoods_df = (neighbourhoods_p
                     .select("id","nhood_name","city_id")
                     .hint("broadcast")
                     )
cities_df   = (cities_p
               .select("id","city_name")
               .hint("broadcast")
               )
listings_df = (listings_p
               .select("id","city_id","neighbourhood","listing_name","minimum_nights")
               )
reviews_df  = reviews_p.select("listing_id","review_date")


select_city_id = (cities_df
                  .filter(F.col("city_name") == city_name)
                  )

listings_select = (listings_df
                .join(select_city_id, listings_df["city_id"] == select_city_id["id"])
                .join(F.broadcast(neighbourhoods_df), listings_df["neighbourhood"] == neighbourhoods_df["id"]) 
                .drop(neighbourhoods_df["city_id"],neighbourhoods_df["id"],select_city_id['id']) 
                .withColumnRenamed("nhood_name", "neighbourhood_name") 
                .withColumnRenamed("id", "listing_id")
)



reviews_2024 = (reviews_df
                .filter(F.year("review_date") == 2024)
                .select("listing_id")  
                .cache()
)

reviews_select_2024 = reviews_2024.join(listings_select, on="listing_id", how="inner")

reviews_select_2024 = (reviews_select_2024
                    .withColumn("stay_length",
                                F.when((F.col("minimum_nights") > 3) & (F.col("minimum_nights") <= 21), F.col("minimum_nights"))
                                .when(F.col("minimum_nights") > 21, F.lit(3))
                                .otherwise(F.lit(3)))
                    .withColumn("occupation_days", F.col("stay_length") * F.lit(2))
                    )


listings_occupation = (listings_select
                       .join(reviews_select_2024
                             .groupBy("listing_id")
                             .agg(F.sum("occupation_days").alias("occupation_days_sum")),
                             on="listing_id",
                             how="left"
                             )
                       .fillna({"occupation_days_sum": 0})
                       .withColumn("occupancy_rate",
                                   F.least(F.col("occupation_days_sum") / F.lit(365), F.lit(1.0))
                                   )
                       )



top10_neigh = (listings_occupation
             .groupBy("neighbourhood_name") 
             .agg(F.avg("occupancy_rate").alias("average_occupancy_rate"))
             .orderBy(F.col("average_occupancy_rate").desc())
             .limit(10)
             )

from pyspark.sql.window import Window


window = Window.partitionBy("neighbourhood_name").orderBy(F.col("occupancy_rate").desc())

top5_grouped = (listings_occupation
                .join(F.broadcast(top10_neigh), on="neighbourhood_name", how="inner")
                .withColumn("rn", F.row_number().over(window))
                .filter(F.col("rn") <= 5)
                .groupBy("neighbourhood_name", "average_occupancy_rate")
                .agg(F.collect_list(F.struct("rn", "listing_name", "occupancy_rate")).alias("listings_struct"))
                .orderBy(F.desc("average_occupancy_rate"),F.desc("neighbourhood_name"))
                )



df1 = top5_grouped.withColumn(
    "pairs",
    F.transform(
        F.col("listings_struct"),
        lambda x: F.concat(
            F.lit("("),
            x["listing_name"],
            F.lit(", "),
            x["occupancy_rate"].cast("string"),
            F.lit(")")
        )
    )
)


df2 = df1.withColumn(
    "listings_str",
    F.concat(
        F.lit("["),
        F.concat_ws(", ", F.col("pairs")),
        F.lit("]")
    )
)

opt_p_3 = df2.withColumn("avg_occ_rate_str", F.format_number(F.col("average_occupancy_rate"), 2))

opt_p_3.select(F.concat_ws("\t", F.col("neighbourhood_name"), F.col("avg_occ_rate_str"), F.col("listings_str")).alias("output")).show(truncate=False)


opt_p_3.explain(True)

stagemetrics.end()
stagemetrics.print_report()


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|output                                                                                                                                                                                                                                                                                                                                  |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Sydney